# Materialized Tables: Neo4j Aircraft Graph as Delta Tables

Materializes the Neo4j aircraft graph as **managed Delta tables** in Unity Catalog. Each node label is read from Neo4j via `remote_query()` and written as a Delta table, making it queryable alongside `sensor_readings` with standard Spark SQL. After materialization, no JDBC or `remote_query()` is needed at query time.

**Why materialize?**
- **Genie / AI agents** can query graph data via natural language
- **Full Spark SQL support** — no `remote_query()` quirks or translator limitations at query time
- **Performance** — Delta tables are optimized for analytical queries and shuffles
- **Governance** — materialized tables appear in Catalog Explorer with full schema and lineage

**Prerequisites:**
- Run `00-load-graph.ipynb` to load the aircraft graph into Neo4j
- Run `01-neo4j-uc-connection-setup.ipynb` to create the UC JDBC connection
- Run `02-federated-queries.ipynb` to create the `sensor_readings` Delta table

## Graph Schema

This notebook materializes the full aircraft graph as Delta tables. Each node label is read from Neo4j via `remote_query()` and written as a managed Delta table. The `HAS_SYSTEM` relationship is materialized explicitly via a `NATURAL JOIN` aggregate (translated to a Cypher `MATCH` pattern by the Neo4j JDBC driver). Other edges are reconstructed at query time using the foreign-key properties stored on each node.

**Nodes (materialized as Delta tables)**

| Label | Delta table | Key props |
|---|---|---|
| Aircraft | neo4j_aircraft | aircraftId, model, manufacturer, operator |
| Airport | neo4j_airports | airportId, iata, city |
| System | neo4j_systems | systemId, aircraftId, type |
| Component | neo4j_components | componentId, systemId, type |
| Sensor | neo4j_sensors | sensorId, systemId, type, unit |
| Flight | neo4j_flights | flightId, aircraftId, operator, origin, destination |
| MaintenanceEvent | neo4j_maintenance_events | eventId, aircraftId, severity, fault |
| Delay | neo4j_delays | delayId, flightId, cause, minutes |

**Relationships**

| Type | From -> To | How represented |
|---|---|---|
| HAS_SYSTEM | Aircraft -> System | `neo4j_aircraft_systems` Delta table (via NATURAL JOIN) + `neo4j_systems.aircraftId` FK |
| HAS_COMPONENT | System -> Component | `neo4j_components.systemId` FK |
| HAS_SENSOR | System -> Sensor | `neo4j_sensors.systemId` FK |
| OPERATES_FLIGHT | Aircraft -> Flight | `neo4j_flights.aircraftId` FK |
| DEPARTS_FROM | Flight -> Airport | `neo4j_flights.origin = neo4j_airports.iata` |
| ARRIVES_AT | Flight -> Airport | `neo4j_flights.destination = neo4j_airports.iata` |
| HAS_DELAY | Flight -> Delay | `neo4j_delays.flightId` FK |
| HAS_EVENT | Component -> MaintenanceEvent | `neo4j_maintenance_events.componentId` FK |

## About materialization

Materialization is the second of the two patterns this project uses. Notebook 02 demonstrated single-statement federation via `remote_query()`; this notebook reads from Neo4j once via `remote_query()` and writes the result as a Delta table.

### Pattern

```python
df = spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => '<SQL string sent to Neo4j>'
    )
""")
df.write.format("delta").mode("overwrite").saveAsTable(f"{FQN}.<table_name>")
```

After the write, the data lives entirely in Unity Catalog. Subsequent queries are pure Spark SQL over Delta — no `remote_query()`, no JDBC, no translator at query time.

### When to materialize vs. when to query live with `remote_query()`

| Need | Use |
|---|---|
| Freshness-sensitive queries (graph changes within minutes matter) | `remote_query()` (notebook 02) |
| Genie / AI agents that need browsable, governed tables | Materialization (this notebook) |
| BI dashboards on a slowly-changing graph | Materialization |
| Ad-hoc exploration | `remote_query()` |
| Heavy joins across many node labels + Delta tables | Materialization (faster after the one-time write) |

Materialization is point-in-time. Re-run this notebook (or schedule Section 2 as a job) to refresh.

### Note on `HAVING COUNT(*) > 0`

The relationship materialization in Section 2 uses `GROUP BY` inside `remote_query()` and includes `HAVING COUNT(*) > 0` to work around a Databricks result-reuse bug. See notebook 02's "How federation works" cell for the full explanation.

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Loaded from Databricks secrets
# =============================================================================

# --- Neo4j Aura ---
SECRET_SCOPE = "neo4j-uc-demos"
NEO4J_URI = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_URI")
NEO4J_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_USERNAME")
NEO4J_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_PASSWORD")

# --- Databricks Unity Catalog ---
UC_CATALOG = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_CATALOG")
UC_SCHEMA = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_SCHEMA")
UC_VOLUME = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_VOLUME")
JDBC_JAR_PATH = dbutils.secrets.get(scope=SECRET_SCOPE, key="JDBC_JAR_PATH")
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_JDBC_URL_SQL = f"jdbc:{NEO4J_URI}/neo4j?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

In [ ]:
from pyspark.sql.types import StringType
import time

def materialize(table_name, query, expected):
    """Read from Neo4j via remote_query() and write as a managed Delta table."""
    df = spark.sql(f"SELECT * FROM remote_query('{UC_CONNECTION_NAME}', query => '{query}')")
    # Cast all columns to STRING for a stable, Delta-friendly schema across runs.
    for c in df.columns:
        df = df.withColumn(c, df[c].cast(StringType()))
    t0 = time.time()
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQN}.{table_name}")
    ms = (time.time() - t0) * 1000
    cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.{table_name}").collect()[0]["cnt"]
    status = "PASS" if cnt == expected else "FAIL"
    print(f"  [{status}] {table_name}: {cnt} rows ({ms:.0f}ms)")

---

## Section 1: Verify Data Sources

Confirms both `sensor_readings` (Delta) and the Neo4j aircraft graph (via `remote_query()`) are accessible before materializing.

In [ ]:
print("--- Section 1: Verify Data Sources ---")

count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.sensor_readings").collect()[0]["cnt"]
status = "PASS" if count == 172800 else "FAIL"
print(f"  [{status}] Delta sensor_readings: {count:,} rows")

# Read each node count through remote_query() against the UC JDBC connection.
for label, expected in [
    ("Aircraft", 20), ("Airport", 12), ("System", 80), ("Component", 320),
    ("Sensor", 160), ("Flight", 800), ("MaintenanceEvent", 300), ("Delay", 514),
]:
    cnt = spark.sql(f"""
        SELECT cnt FROM remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT COUNT(*) AS cnt FROM {label}'
        )
    """).collect()[0]["cnt"]
    status = "PASS" if cnt == expected else "FAIL"
    print(f"  [{status}] Neo4j {label}: {cnt} nodes")

---

## Section 2: Materialize Neo4j Nodes as Delta Tables

Reads each node label from Neo4j via `remote_query()` and writes it as a managed Delta table. Also materializes the Aircraft → System relationship via a `NATURAL JOIN` aggregate (translated to a Cypher `MATCH` traversal by the Neo4j JDBC driver).

In [ ]:
print("--- Section 2: Materialize Neo4j Data ---")

materialize("neo4j_aircraft",
    "SELECT aircraftId, tail_number, icao24, model, manufacturer, operator FROM Aircraft",
    20)

materialize("neo4j_airports",
    "SELECT airportId, name, city, country, iata, icao FROM Airport",
    12)

materialize("neo4j_systems",
    "SELECT systemId, aircraftId, type, name FROM System",
    80)

materialize("neo4j_sensors",
    "SELECT sensorId, systemId, type, name, unit FROM Sensor",
    160)

materialize("neo4j_components",
    "SELECT componentId, systemId, type, name FROM Component",
    320)

materialize("neo4j_maintenance_events",
    "SELECT eventId, componentId, systemId, aircraftId, fault, severity, reported_at, corrective_action FROM MaintenanceEvent",
    300)

materialize("neo4j_flights",
    "SELECT flightId, flight_number, aircraftId, operator, origin, destination, scheduled_departure, scheduled_arrival FROM Flight",
    800)

materialize("neo4j_delays",
    "SELECT delayId, flightId, cause, CAST(minutes AS STRING) AS minutes FROM Delay",
    514)

# Aircraft -> HAS_SYSTEM -> System relationship materialized as a Delta table.
# The NATURAL JOIN chain is translated to a Cypher MATCH traversal on the Neo4j side.
# HAVING COUNT(*) > 0 is the remote_query() GROUP BY workaround (see note near the top).
df = spark.sql(f"""
    SELECT aircraftId, model, systemId, systemType, systemName
    FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT a.aircraftId AS aircraftId,
                         a.model AS model,
                         s.systemId AS systemId,
                         s.type AS systemType,
                         s.name AS systemName,
                         COUNT(*) AS cnt
                  FROM Aircraft a NATURAL JOIN HAS_SYSTEM rel NATURAL JOIN System s
                  GROUP BY a.aircraftId, a.model, s.systemId, s.type, s.name
                  HAVING COUNT(*) > 0'
    )
""")

for c in df.columns:
    df = df.withColumn(c, df[c].cast(StringType()))
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQN}.neo4j_aircraft_systems")
cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.neo4j_aircraft_systems").collect()[0]["cnt"]
status = "PASS" if cnt == 80 else "FAIL"
print(f"  [{status}] neo4j_aircraft_systems: {cnt} rows")

---

## Section 3: Verify UC Schema Tracking

When `saveAsTable()` writes a Delta table, Unity Catalog automatically registers its full
schema: column names, types, nullability, and statistics, making it browsable in
**Catalog Explorer** and queryable via `INFORMATION_SCHEMA`. No separate metadata sync
step is required: the writes in Section 2 are also the metadata sync.

In [ ]:
print("--- Section 3: Verify UC Schema Tracking ---")

# All neo4j_* tables registered in INFORMATION_SCHEMA.TABLES
print("\n  Tables registered in UC:")
spark.sql(f"""
    SELECT table_name, table_type
    FROM `{UC_CATALOG}`.information_schema.tables
    WHERE table_schema = '{UC_SCHEMA}'
      AND table_name LIKE 'neo4j_%'
    ORDER BY table_name
""").show(truncate=False)

# Column-level schema for one representative table
print("  Columns tracked for neo4j_aircraft:")
spark.sql(f"""
    SELECT ordinal_position, column_name, data_type, is_nullable
    FROM `{UC_CATALOG}`.information_schema.columns
    WHERE table_schema = '{UC_SCHEMA}'
      AND table_name = 'neo4j_aircraft'
    ORDER BY ordinal_position
""").show(truncate=False)

print("  All tables are browsable in Catalog Explorer with full schema and lineage.")

---

## Section 4: SQL Validation Tests

Verifies that standard SQL operations work on the materialized Delta tables.

In [ ]:
print("--- Section 4: SQL Validation Tests ---")

# TEST 1: GROUP BY: maintenance events by severity
print("\n  TEST 1: GROUP BY, maintenance events by severity")
spark.sql(f"""
    SELECT severity, COUNT(*) AS event_count
    FROM {FQN}.neo4j_maintenance_events
    GROUP BY severity
    ORDER BY event_count DESC
""").show(truncate=False)

# TEST 2: WHERE + ORDER BY: critical maintenance events
print("  TEST 2: WHERE + ORDER BY, critical events")
spark.sql(f"""
    SELECT eventId, aircraftId, fault, reported_at
    FROM {FQN}.neo4j_maintenance_events
    WHERE severity = 'CRITICAL'
    ORDER BY reported_at DESC
""").show(10, truncate=False)

# TEST 3: Aggregations + JOIN: sensor count per aircraft model
print("  TEST 3: Aggregations + JOIN, sensor count per model")
spark.sql(f"""
    SELECT a.model, a.manufacturer,
           COUNT(DISTINCT s.sensorId) AS sensor_count,
           COUNT(DISTINCT sys.systemId) AS system_count
    FROM {FQN}.neo4j_aircraft a
    JOIN {FQN}.neo4j_systems sys ON a.aircraftId = sys.aircraftId
    JOIN {FQN}.neo4j_sensors s ON sys.systemId = s.systemId
    GROUP BY a.model, a.manufacturer
    ORDER BY sensor_count DESC
""").show(truncate=False)

# TEST 4: DISTINCT: unique manufacturers and models
print("  TEST 4: DISTINCT, manufacturers and models")
spark.sql(f"""
    SELECT DISTINCT manufacturer, model
    FROM {FQN}.neo4j_aircraft
    ORDER BY manufacturer, model
""").show(truncate=False)

---

## Section 5: Analytics on Materialized Tables

Joins the materialized `neo4j_*` tables with `sensor_readings` in pure Spark SQL — no `remote_query()` or JDBC needed at query time. After materialization, Neo4j-sourced data behaves like any other Delta table in Unity Catalog.

In [ ]:
print("--- Section 5: Analytics on Materialized Tables ---")

# Query 1: Aircraft health overview
print("\n  Query 1: Aircraft Health Overview")
spark.sql(f"""
    SELECT a.aircraftId, a.model, a.operator,
           COUNT(DISTINCT m.eventId) AS maintenance_events,
           COUNT(DISTINCT CASE WHEN m.severity = 'CRITICAL' THEN m.eventId END) AS critical_events,
           COUNT(DISTINCT s.sensorId) AS sensor_count,
           ROUND(AVG(r.value), 2) AS avg_sensor_reading
    FROM {FQN}.neo4j_aircraft a
    LEFT JOIN {FQN}.neo4j_maintenance_events m ON a.aircraftId = m.aircraftId
    LEFT JOIN {FQN}.neo4j_sensors s ON s.sensorId LIKE CONCAT(a.aircraftId, '-%')
    LEFT JOIN {FQN}.sensor_readings r ON r.sensorId = s.sensorId
    GROUP BY a.aircraftId, a.model, a.operator
    ORDER BY critical_events DESC, maintenance_events DESC
""").show(10, truncate=False)

# Query 2: Route analysis, busiest airport pairs
print("\n  Query 2: Route Analysis")
spark.sql(f"""
    SELECT dep.city AS origin_city, dep.iata AS origin,
           arr.city AS destination_city, arr.iata AS destination,
           COUNT(*) AS flight_count
    FROM {FQN}.neo4j_flights f
    JOIN {FQN}.neo4j_airports dep ON f.origin = dep.iata
    JOIN {FQN}.neo4j_airports arr ON f.destination = arr.iata
    GROUP BY dep.city, dep.iata, arr.city, arr.iata
    ORDER BY flight_count DESC
    LIMIT 10
""").show(truncate=False)

# Query 3: Sensor health by system type
print("\n  Query 3: Sensor Health by System Type")
spark.sql(f"""
    SELECT sys.type AS system_type,
           COUNT(DISTINCT s.sensorId) AS sensor_count,
           COUNT(r.readingId) AS total_readings,
           ROUND(AVG(r.value), 2) AS avg_reading,
           ROUND(STDDEV(r.value), 2) AS stddev_reading
    FROM {FQN}.neo4j_sensors s
    JOIN {FQN}.neo4j_systems sys ON s.systemId = sys.systemId
    JOIN {FQN}.sensor_readings r ON r.sensorId = s.sensorId
    GROUP BY sys.type
    ORDER BY total_readings DESC
""").show(truncate=False)

# Query 4: Delay analysis with airport and operator info
print("\n  Query 4: Delay Analysis by Airport and Cause")
spark.sql(f"""
    SELECT dep.city AS departure_city, dep.iata,
           d.cause, COUNT(*) AS delay_count,
           ROUND(AVG(CAST(d.minutes AS INT)), 1) AS avg_delay_minutes
    FROM {FQN}.neo4j_delays d
    JOIN {FQN}.neo4j_flights f ON d.flightId = f.flightId
    JOIN {FQN}.neo4j_airports dep ON f.origin = dep.iata
    GROUP BY dep.city, dep.iata, d.cause
    ORDER BY delay_count DESC
    LIMIT 15
""").show(truncate=False)

print("\nStatus: PASS")

---

## Summary

All Neo4j graph data is now available as managed Delta tables in Unity Catalog.

| Table | Source | Rows |
|-------|--------|------|
| `neo4j_aircraft` | Aircraft nodes | 20 |
| `neo4j_airports` | Airport nodes | 12 |
| `neo4j_systems` | System nodes | 80 |
| `neo4j_sensors` | Sensor nodes | 160 |
| `neo4j_components` | Component nodes | 320 |
| `neo4j_maintenance_events` | MaintenanceEvent nodes | 300 |
| `neo4j_flights` | Flight nodes | 800 |
| `neo4j_delays` | Delay nodes | 514 |
| `neo4j_aircraft_systems` | Aircraft → HAS_SYSTEM → System | 80 |

**UC tracks the schema automatically:**
- All tables appear in Catalog Explorer with column names, types, and lineage
- Schema is queryable via `INFORMATION_SCHEMA`. No separate metadata sync step needed.

**Next steps:**
- Add `sensor_readings` and all `neo4j_*` tables to a Genie space for natural language queries
- Schedule notebook Section 2 to refresh Neo4j data periodically